# Model Evaluation & Comparison
This notebook evaluates all trained RL models (PPO, A2C, SAC, TD3) across multiple seeds and compares their performance using key metrics.

In [ ]:
#  IMPORT LIBRARIES
import json
import os
from statistics import mean, pstdev

# Reinforcement Learning models
from stable_baselines3 import PPO, A2C, SAC, TD3

# Custom environment
from env.lighting_env import SmartLightingEnv

#  CONFIGURATION

In [ ]:
# Supported algorithms
MODEL_CLASSES = {
    "PPO": PPO,
    "A2C": A2C,
    "SAC": SAC,
    "TD3": TD3,
}

# Seeds used during training
SEEDS = [1, 2, 3]

# Directory storing trained models
RESULTS_DIRECTORY = "results_train"

# Number of evaluation episodes
EVAL_EPISODES = 10

#  EVALUATE MODEL

In [ ]:
def evaluate_model_file(model_class, model_path, number_of_episodes=10):
    """
    Evaluate a trained model across multiple episodes.
    Returns averaged performance metrics.
    """

    trained_model = model_class.load(model_path)

    cumulative_reward_list = []
    comfort_list = []
    energy_list = []
    colour_list = []
    overall_list = []
    illuminance_error_list = []
    power_list = []

    for episode_index in range(number_of_episodes):

        env = SmartLightingEnv()
        observation, _ = env.reset(seed=episode_index)

        terminated = False
        truncated = False

        episode_reward = 0.0
        comfort_scores = []
        energy_scores = []
        colour_scores = []
        overall_scores = []
        illuminance_errors = []
        power_values = []

        while not (terminated or truncated):
            action, _ = trained_model.predict(observation, deterministic=True)
            observation, reward, terminated, truncated, info = env.step(action)

            episode_reward += float(reward)
            comfort_scores.append(float(info["comfort_score"]))
            energy_scores.append(float(info["energy_efficiency_score"]))
            colour_scores.append(float(info["colour_temperature_score"]))
            overall_scores.append(float(info["overall_score"]))
            illuminance_errors.append(float(info["illuminance_error"]))
            power_values.append(float(info["power_consumption"]))

        cumulative_reward_list.append(episode_reward)
        comfort_list.append(mean(comfort_scores))
        energy_list.append(mean(energy_scores))
        colour_list.append(mean(colour_scores))
        overall_list.append(mean(overall_scores))
        illuminance_error_list.append(mean(illuminance_errors))
        power_list.append(mean(power_values))

        env.close()

    return {
        "reward": mean(cumulative_reward_list),
        "comfort": mean(comfort_list),
        "energy": mean(energy_list),
        "colour": mean(colour_list),
        "overall": mean(overall_list),
        "error": mean(illuminance_error_list),
        "power": mean(power_list),
    }

# Model Path Functions

In [ ]:
def get_final_model_path(algorithm_name, seed):
    return os.path.join(
        RESULTS_DIRECTORY,
        algorithm_name.lower(),
        f"run_{seed}",
        f"{algorithm_name.lower()}_final_seed_{seed}.zip",
    )

def get_best_model_path(algorithm_name, seed):
    return os.path.join(
        RESULTS_DIRECTORY,
        algorithm_name.lower(),
        f"run_{seed}",
        "best_model",
        "best_model.zip",
    )

def resolve_model_path(algorithm_name, seed):
    best_model_path = get_best_model_path(algorithm_name, seed)
    final_model_path = get_final_model_path(algorithm_name, seed)

    if os.path.exists(best_model_path):
        return best_model_path, "best_model"
    if os.path.exists(final_model_path):
        return final_model_path, "final_model"

    return None, None

# Multi-run Evaluation

In [ ]:
def evaluate_algorithm_multi_run(algorithm_name):

    model_class = MODEL_CLASSES[algorithm_name]
    run_results = []

    for seed in SEEDS:
        model_path, source_type = resolve_model_path(algorithm_name, seed)

        if not model_path:
            print(f"{algorithm_name} run with seed={seed} not found.")
            continue

        metrics = evaluate_model_file(model_class, model_path, number_of_episodes=EVAL_EPISODES)

        metrics["seed"] = seed
        metrics["path"] = model_path
        metrics["source_type"] = source_type

        run_results.append(metrics)

    if not run_results:
        return None

    best_run = max(run_results, key=lambda item: item["overall"])

    return {
        "name": algorithm_name,
        "runs": run_results,
        "reward_mean": mean(item["reward"] for item in run_results),
        "reward_std": pstdev(item["reward"] for item in run_results),
        "comfort_mean": mean(item["comfort"] for item in run_results),
        "comfort_std": pstdev(item["comfort"] for item in run_results),
        "energy_mean": mean(item["energy"] for item in run_results),
        "energy_std": pstdev(item["energy"] for item in run_results),
        "colour_mean": mean(item["colour"] for item in run_results),
        "colour_std": pstdev(item["colour"] for item in run_results),
        "overall_mean": mean(item["overall"] for item in run_results),
        "overall_std": pstdev(item["overall"] for item in run_results),
        "error_mean": mean(item["error"] for item in run_results),
        "power_mean": mean(item["power"] for item in run_results),
        "best_seed": best_run["seed"],
        "best_overall": best_run["overall"],
        "best_path": best_run["path"],
    }

# Run Evaluation

In [ ]:
results = []

for algorithm_name in MODEL_CLASSES:
    result = evaluate_algorithm_multi_run(algorithm_name)
    if result:
        results.append(result)

if not results:
    print("No models found.")

# Display Table + Ranking

In [ ]:
# Sort by performance
results = sorted(results, key=lambda r: r["overall_mean"], reverse=True)

print("\n===== MULTI-RUN MODEL COMPARISON RESULTS =====\n")

# Table header
print(
    f"{'Rank':<5} | {'Algorithm':<8} | {'Reward':>18} | {'Comfort':>18} | "
    f"{'Energy':>18} | {'Colour':>18} | {'Overall':>18} | {'Best Seed':>10}"
)
print("-" * 140)

# Table rows
for i, result in enumerate(results, start=1):
    print(
        f"{i:<5} | "
        f"{result['name']:<8} | "
        f"{result['reward_mean']:>8.2f} ± {result['reward_std']:<7.2f} | "
        f"{result['comfort_mean']:>8.2f} ± {result['comfort_std']:<7.2f} | "
        f"{result['energy_mean']:>8.2f} ± {result['energy_std']:<7.2f} | "
        f"{result['colour_mean']:>8.2f} ± {result['colour_std']:<7.2f} | "
        f"{result['overall_mean']:>8.2f} ± {result['overall_std']:<7.2f} | "
        f"{result['best_seed']:>10}"
    )

# Best model
best_model = results[0]
print(f"\n🏆 BEST MODEL: {best_model['name']} (Score = {best_model['overall_mean']:.2f})")


===== MULTI-RUN MODEL COMPARISON RESULTS =====

Rank  | Algorithm |             Reward |            Comfort |             Energy |             Colour |            Overall |  Best Seed
--------------------------------------------------------------------------------------------------------------------------------------------
1     | PPO      |    52.01 ± 0.08    |    93.09 ± 0.34    |    99.56 ± 0.09    |    60.41 ± 6.83    |    88.17 ± 1.25    |          3
2     | TD3      |    51.83 ± 0.19    |    94.20 ± 0.10    |    98.21 ± 0.10    |    52.16 ± 7.86    |    86.80 ± 1.61    |          2
3     | SAC      |    52.19 ± 0.02    |    94.06 ± 0.11    |    94.70 ± 0.23    |    48.93 ± 2.01    |    85.19 ± 0.42    |          2
4     | A2C      |    42.93 ± 0.38    |    67.58 ± 1.16    |    89.46 ± 1.15    |    31.68 ± 8.86    |    65.87 ± 1.57    |          3

🏆 BEST MODEL: PPO (Score = 88.17)


In [ ]:
print("\n--- Per-run details ---\n")

for result in results:
    print(f"{result['name']}:")

    for run in result["runs"]:
        best_marker = "  <-- BEST" if run["seed"] == result["best_seed"] else ""
        print(
            f"  seed={run['seed']} | "
            f"reward={run['reward']:.2f} | "
            f"comfort={run['comfort']:.2f} | "
            f"energy={run['energy']:.2f} | "
            f"colour={run['colour']:.2f} | "
            f"overall={run['overall']:.2f} | "
            f"source={run['source_type']}{best_marker}"
        )

    print(f"  Mean illuminance error: {result['error_mean']:.2f} lux")
    print(f"  Mean power consumption: {result['power_mean']:.10f}")
    print(f"  Best seed: {result['best_seed']}")
    print(f"  Best path: {result['best_path']}")
    print()


--- Per-run details ---

PPO:
  seed=1 | reward=51.91 | comfort=93.50 | energy=99.67 | colour=51.35 | overall=86.61 | source=best_model
  seed=2 | reward=52.03 | comfort=92.67 | energy=99.44 | colour=62.03 | overall=88.23 | source=best_model
  seed=3 | reward=52.11 | comfort=93.10 | energy=99.56 | colour=67.85 | overall=89.66 | source=best_model  <-- BEST
  Mean illuminance error: 57.69 lux
  Mean power consumption: 0.2244803311
  Best seed: 3
  Best path: results_train\ppo\run_3\best_model\best_model.zip

TD3:
  seed=1 | reward=51.73 | comfort=94.14 | energy=98.16 | colour=43.29 | overall=84.98 | source=best_model
  seed=2 | reward=52.10 | comfort=94.34 | energy=98.12 | colour=62.40 | overall=88.90 | source=best_model  <-- BEST
  seed=3 | reward=51.66 | comfort=94.12 | energy=98.36 | colour=50.80 | overall=86.52 | source=best_model
  Mean illuminance error: 49.02 lux
  Mean power consumption: 0.2592098042
  Best seed: 2
  Best path: results_train\td3\run_2\best_model\best_model.zip

